In [2]:
import pandas as pd
pd.set_option('max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import scipy.stats as stats
sns.set()

%matplotlib inline 


# import sklearn
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
confusion_matrix, f1_score, roc_auc_score, make_scorer)
from sklearn.model_selection import GridSearchCV 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_csv('data/allegations_202007271729.csv')
df.head().T

,0,1,2,3,4
unique_mos_id,10004,10007,10007,10007,10009
first_name,Jonathan,John,John,John,Noemi
last_name,Ruiz,Sears,Sears,Sears,Sierra
command_now,078 PCT,078 PCT,078 PCT,078 PCT,078 PCT
shield_no,8409,5952,5952,5952,24058
complaint_id,42835,24601,24601,26146,40253
month_received,7,11,11,7,8
year_received,2019,2011,2011,2012,2018
month_closed,5,8,8,9,2
year_closed,2020,2012,2012,2013,2019


In [ ]:
df['contact_reason'].value_counts()

In [ ]:
df['allegation'].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df['rank_incident'].value_counts()

In [ ]:
df['outcome_description'].value_counts()

In [ ]:
df['board_disposition'].value_counts()

## Drop columns that will not be useful for model 
Not going to use `first_name`, `shiedl_no`, `last_name`, `rank_now`, `rank_incident`. Removing the last two because they're redundant and rank_abbrev have more detail. 

In [ ]:
df.drop(['first_name', 'last_name', 'rank_incident', 'rank_now', 'shield_no'], axis=1, inplace=True)

In [ ]:
#Sanity check for modified df
df.head().T

In [ ]:
# check for duplicates 
df.duplicated().sum()

In [ ]:
# Remove duplicate rows 
print(f'Shape before removing duplicates: {df.shape}')
df.drop_duplicates(inplace=True)
# Sanity check 
print(f'Num duplicates remaining: {df.duplicated().sum()}')
print(f'Shape after removing duplicates: {df.shape}')

In [ ]:
# Sort df by date 
df.sort_values(by=['year_received', 'month_received'], inplace=True, ascending=True)

In [ ]:
# Sanity check 
df.head().T

In [ ]:
print(df['precinct'].value_counts(bins=5))

In [ ]:
# create column for unsub/exonerated 
condition = (df['board_disposition']=='Unsubstantiated') | (df['board_disposition']=='Exonerated')
df['unsubst_or_exonerated'] = np.where(condition, 1, 0)

In [ ]:
counts = df['precinct'].value_counts()
counts.plot(kind='barh', figsize=(8, 13))

In [5]:
df['precinct'] = df['precinct'].fillna(-999)

In [6]:
manhattan =[1, 5, 6, 7, 9, 10, 13, 17, 19, 20, 23, 24, 25, 26, 28, 30, 32, 33, 34]
bronx = [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52]
brooklyn = [60, 61, 62, 63, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 81, 83, 84, 88, 90, 94]
queens = [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
staten = [120, 121, 122, 123]
def my_function(x):
    if int(x) in (manhattan):
        return 'Manhattan'
    elif int(x) in (bronx):
        return 'Bronx'
    elif int(x) in (brooklyn):
        return 'Brooklyn'
    elif int(x) in (queens):
        return 'Queens'
    elif int(x) in (staten):
        return 'Staten Island'
    else:
        return 'Other'
    
df['Boroughs'] = df.precinct.apply(my_function)
df.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,command_at_incident,rank_abbrev_incident,rank_abbrev_now,rank_now,rank_incident,mos_ethnicity,mos_gender,mos_age_incident,complainant_ethnicity,complainant_gender,complainant_age_incident,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,Boroughs
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,078 PCT,POM,POM,Police Officer,Police Officer,Hispanic,M,32,Black,Female,38.0,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn
1,10007,John,Sears,078 PCT,5952,24601,11,2011,8,2012,PBBS,POM,POM,Police Officer,Police Officer,White,M,24,Black,Male,26.0,Discourtesy,Action,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),Brooklyn
2,10007,John,Sears,078 PCT,5952,24601,11,2011,8,2012,PBBS,POM,POM,Police Officer,Police Officer,White,M,24,Black,Male,26.0,Offensive Language,Race,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),Brooklyn
3,10007,John,Sears,078 PCT,5952,26146,7,2012,9,2013,PBBS,POM,POM,Police Officer,Police Officer,White,M,25,Black,Male,45.0,Abuse of Authority,Question,67.0,PD suspected C/V of violation/crime - street,No arrest made or summons issued,Substantiated (Charges),Brooklyn
4,10009,Noemi,Sierra,078 PCT,24058,40253,8,2018,2,2019,078 PCT,POF,POF,Police Officer,Police Officer,Hispanic,F,39,NaN,NaN,16.0,Force,Physical force,67.0,Report-dispute,Arrest - other violation/crime,Substantiated (Command Discipline A),Brooklyn


In [ ]:
df['Boroughs'].value_counts()

In [ ]:
c = df['Boroughs'].value_counts()
c.plot(kind='bar', figsize=(13, 8))

In [ ]:
df['year_received'].value_counts()

In [ ]:
c1 = df['Boroughs'] == 'Brooklyn'
c2 = df['year_received'] == 2016
df['condition1'] = np.where(c1 & c2, 1, 0)
a = len(df[df['condition1'] == 1])

c3 = df['Boroughs'] == 'Bronx'
c4 = df['year_received'] == 2016
df['condition2'] = np.where(c3 & c4, 1, 0)
b = len(df[df['condition2'] == 1])

c5 = df['Boroughs'] == 'Manhattan'
c6 = df['year_received'] == 2016
df['condition3'] = np.where(c5 & c6, 1, 0)
c = len(df[df['condition3'] == 1])

c7 = df['Boroughs'] == 'Queens'
c8 = df['year_received'] == 2016
df['condition4'] = np.where(c7 & c8, 1, 0)
d = len(df[df['condition4'] == 1])

c9 = df['Boroughs'] == 'Staten Island'
c10 = df['year_received'] == 2016
df['condition5'] = np.where(c9 & c10, 1, 0)
e = len(df[df['condition5'] == 1])

c11 = df['Boroughs'] == 'Other'
c12 = df['year_received'] == 2016
df['condition6'] = np.where(c11 & c12, 1, 0)
f = len(df[df['condition6'] == 1])

print(a+b+c+d+e+f)
print(a)

In [ ]:
def returnSum(borough, year):
    c1 = df['Boroughs'] == borough
    c2 = df['year_received'] == year
    df['condition1'] = np.where(c1 & c2, 1, 0)
    return len(df[df['condition1'] == 1])

borough = input("Enter the borough: ")
year = int(input("Enter the year: "))
returnSum(borough, year)

# df['mos_ethnicity'].value_counts()
df['mos_gender'].value_counts()

In [ ]:
sns.heatmap(df.corr(), annot=True, cmap='Spectral')

In [ ]:
c = df['Boroughs'].value_counts()
e = df['mos_ethnicity'].value_counts()
df.plot(kind='scatter', x='')

In [ ]:
df.head().T

In [ ]:
df.isnull().sum()

In [ ]:
df['mos_gender'].isnull().sum()

## Split our data into training and testing sets. 
The data will be split based on date. 

In [ ]:
# Build a dataframe with the years and how many times they occur in the complaints df
year_val_counts = df['year_received'].value_counts()
df_year_val_counts = pd.DataFrame(year_val_counts)
df_year_val_counts = df_year_val_counts.reset_index()
df_year_val_counts.columns = ['year', 'count']
df_year_val_counts.sort_values(by=['year'], ascending=True, inplace=True)

In [ ]:
df_year_val_counts

In [ ]:
df_year_val_counts['count'].sum()

## Given data distribution, will create a split the data at 2016 
All complaints starting 2017 will be reserved for test data set. 

In [ ]:
# Create a dataframe with training data
df_train = df[df['year_received'] < 2017]
df_test = df[df['year_received'] > 2016] 

In [ ]:
# Sanity checks 
df_train.head()

In [ ]:
print(f'Earliest year in training dataframe: {df_train["year_received"].min()} \nLatest year in training dataframe: {df_train["year_received"].max()}')
print(f'Earliest year in testing dataframe: {df_test["year_received"].min()} \nLatest year in testing dataframe: {df_test["year_received"].max()}')

In [ ]:
print(f'Training dataframe shape: {df_train.shape} \nTesting dataframe shape: {df_test.shape}')
# make sure dataframe samples add up to the total of the original dataframe
print(df_train.shape[0]+df_test.shape[0]==df.shape[0])

## Transform and prep data in each dataframe for modeling 

In [ ]:
# Check datatypes 
df_train.info()

In [ ]:
# check for null values 
df_train.isna().sum()

In [ ]:
df_train['complainant_age_incident'].value_counts()

In [ ]:
def prep_data(df):
    """
    Transforms data to suit modeling needs. Returns df. 
    """
    # Fill null values in numeric rows with median to avoid impact of outliers 
    for label, content in df.items():
        if pd.api.types.is_numeric_dtype(content):
            if pd.isnull(content).sum():
                # Add a column to indicate if data was missing
                df[label+"_is_missing"] = pd.isnull(content)
                # Fill missing numeric values with median
                df[label] = content.fillna(content.median())
    
        # Turn object/string rows into categories 
        if not pd.api.types.is_numeric_dtype(content):
            # Column to indicate if data was missing
            df[label+"_is_missing"] = pd.isnull(content)
            # add +1 to the category code because pandas encodes missing vals with -1
            # This way NaN values will be codes as 1 
            df[label] = pd.Categorical(content).codes+1
    
    return df

In [ ]:
# Use prep_data function to categories and fill NaN vals in training df
df_train = prep_data(df_train)

In [ ]:
# Use prep_data function to categories and fill NaN vals in testing df
df_test = prep_data(df_test)

## Create and evaluate random forest classification model

In [ ]:
df_train.head().T

In [ ]:
X_train, y_train = df_train.drop(['outcome_description', 'board_disposition', 'unsubst_or_exonerated'], axis=1), df_train['unsubst_or_exonerated']
X_test, y_test = df_test.drop(['outcome_description', 'board_disposition', 'unsubst_or_exonerated'], axis=1), df_test['unsubst_or_exonerated']

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
def evaluate_class_model(model): 
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_true=y_test, y_pred=y_pred, )
    print("Accuracy Score: %f" % accuracy)

    precision = precision_score(y_true=y_test, y_pred=y_pred, average='micro')
    print("Precision Score: %f" % precision)

    recall = recall_score(y_true=y_test, y_pred=y_pred, average='micro')
    print("Recall Score: %f" % recall)

    f1 = f1_score(y_true=y_test, y_pred=y_pred, average='micro')
    print('F1 Score: %f' % f1)

#     Calculate predicted probabilities, keep only probability for when class = 1
    y_pred_proba = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_true=y_test, y_score=y_pred_proba, multi_class='ovo')
    print(f'AUC Score: %f' % auc)

In [ ]:
model = RandomForestClassifier(n_estimators=20, criterion='gini', max_features=25, 
                              min_samples_split=2, max_depth=100)
model.fit(X_train, y_train)

In [ ]:
evaluate_class_model(model)

In [ ]:
params= {'criterion': ['gini', 'entropy'], 
         'max_depth': [2, 4, 5, 10, 20, 100], 
         'max_features': [2, 5, 10, 15, 25, 'auto'], 
         'min_samples_split': [2, 10, 20, 100], 
         'n_estimators': [2, 5, 10, 20, 100],
         }


grid_search_cv =  GridSearchCV( 
    estimator=RandomForestClassifier(), 
    param_grid = params, 
    scoring = 'f1'
    )

In [ ]:
grid_search_cv.fit(X_train, y_train)

In [ ]:
print(grid_search_cv.best_params_)

In [ ]:
model = grid_search_cv.best_estimator_
evaluate_class_model(model)

In [ ]:
# check feature importance 
selected_features = list(df_train.drop(['outcome_description', 'board_disposition', 'unsubst_or_exonerated'], axis=1))
feature_imp = pd.DataFrame.from_dict( {'feature_importance': model.feature_importances_,
                                       'feature':selected_features }).sort_values('feature_importance', ascending=False)
feature_imp

In [ ]:
import pickle
# Dump the trained decision tree classifier with Pickle
filename = 'rf_classifier.pkl'
# Open the file to save as pkl file
pickle.dump(model, open(filename, 'wb'))

In [ ]:
df.value_counts()